In [ ]:
x = 5
print(x)

In [ ]:
#Llamar a 2 librerias
import os
import pandas as pd
#Como trabajar con archivos de una carpeta, es tener un path con la carpeta
ruta_carpeta = r"C:\PCAD"
#Cargar los 3 Dataframes
january_2019 = pd.read_csv(os.path.join(ruta_carpeta,"January 2019.csv"))
february_2019 = pd.read_csv(os.path.join(ruta_carpeta,"February 2019.csv"))
march_2019 = pd.read_csv(os.path.join(ruta_carpeta,"March 2019.csv"))
#Vamos a concatenar los 3 dataframes para tener 1 solo con los
#3 meses
trimestre_2019 = pd.concat([january_2019,february_2019,march_2019])
trimestre_2019.head(20)    #Ver los primeros 20 registros
trimestre_2019.tail(20)    #Ver los ultimos 20 registros
#Ver información del dataframe
trimestre_2019.info()
#Almacenar en una variable la suma de todas las filas del campo revenue
total_revenue = trimestre_2019['Revenue'].sum()
print()   #linea en blanco
print("la suma de revenue es",total_revenue)
#Vamos a filtrar por un campo (Country)
filtro_Canada = trimestre_2019['Country']=="Canada"
trimestre_2019[filtro_Canada]
#Vamos a filtrar por mas de un valor, por Canada y USA usaremos
#el operador de logica or = | (AltGr + 1)
filtro_Canada_USA = (trimestre_2019['Country']=="Canada")|(trimestre_2019['Country']=="USA")
trimestre_2019[filtro_Canada_USA]
#Averiguar el numero de pedidos en Canada + USA
pedidos_CANUSA = trimestre_2019[filtro_Canada_USA]['IdOrder'].count()
print("El numero de pedidos en USA+Canada es",pedidos_CANUSA)
#El promedio de unidades campo Units siempre y cuando el valor de revenue sea mayor o igual de 3000
filtro_m3k = trimestre_2019['Revenue']>=3000
promedio_m3k = round(trimestre_2019[filtro_m3k].Units.mean(),2)
print("El promedio de Units para Revenue >=3000 es",promedio_m3k)
#El operador de logica and (& = Shit/Mayus + 6)

Operadores de comparación en Python
> mayor que
>= mayor o igual que
< menor que
<= menor o igual que
== igual que
!= distinto que

Practica 1: Aprovechando el DataFrame de trimestre_2019, crear una columna nueva (Import Discount) que contenga el importe de descuento que equivale al 5,67% solo para los pedidos cuyo campo Units sea mayor de 20.
Calcular la suma de Import Discount para los pedidos cuyo campo revenue este comprendido entre 3000 y 12000.

In [ ]:
#Solución practica 1
#Voy a crear un dataframe con los datos filtrados para hacer el descuento
trimestre_2019_UU20 = trimestre_2019[trimestre_2019['Units']>20]
trimestre_2019_UU20['Import Discount'] = round(trimestre_2019_UU20['Revenue'] * (1-0.0567),2)
#Para aplicar el filtro contrario con la virgulilla Alt+126
#el 126 del teclado numerico
trimestre_2019_UD20 = trimestre_2019[trimestre_2019['Units']<=20]
trimestre_2019_UD20['Import Discount'] = 0
trimestre_2019_UD20

trimestre_2019 = pd.concat([trimestre_2019_UU20,trimestre_2019_UD20])
#Aplicar un filtro para los comprendidos entre 3000-12000
filtro_312K = (trimestre_2019['Revenue']>=3000) & (trimestre_2019['Revenue']<=12000)
suma_pedidos = trimestre_2019[filtro_312K]['Import Discount'].sum()
print("El resultado es",suma_pedidos)

El modulo Polars (Pandas vitaminado), tips velocidad, eficiencia de memoria y paralelismo. ¿Por que es potente?
- Esta escrito en Rust (lenguaje muy rapido)
- Menos errores de memoria, ejecución muy rapida
- Puede trabajar con varios nucleos de la CPU
- Gestiona bien la RAM
- Usa formato columnar (no filas)
- Se basa en Apache Arrow
- paralelismo automatico (pandas usa 1 nucleo)
- Pandas tarda 4 segundos, en polars tarda 0,5 segundos
- El modo lazy = escribes la instruccion pero no se ejecuta, hasta que le das otra orden

In [ ]:
#para leer de un fichero de excel, necesito tener instalada la libreria openpyxl
#!pip install openpyxl
import pandas as pd
ruta_fichero = r"C:\PCAD\Datos Telefonia Separados Meses Comerciales URL.xlsx"
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_enero
#Como eliminar las filas en blanco, que todos sus campos estan en blanci
fras_enero = fras_enero.dropna(how='all')
fras_enero
#fras_enero.dropna() = eliminar filas con al menos un campo vacio
#fras_enero.dropna(subset=['Nombre campo']) donde el campo nulo
#en la misma linea que leo puedo filtrar
#fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2).dropna(how='all')

In [ ]:
#Como ver los valores unicos de un campo
fras_enero['Servicios'].unique()

para trabajar con fechas en pandas, usaremos el accesor dt
dt = datetime (fecha + hora)

In [ ]:
from datetime import date   #importo la clase date de la libreria datetime = solo me permite trabajar con fecha
hoy = date.today()
#hoy es una instancia de la clase date, hereda los atributos + metodos
print(hoy)
#para ver dia y ahora la clase datetime
from datetime import datetime as dt
hoy2 = dt.now()
print("La fecha + hora actual es",hoy2)

In [ ]:
print("El dia de hoy es",hoy.day)
print("El mes de hoy es",hoy.month)
print("El año de hoy es",hoy.year)

In [ ]:
#Para ajustar una fecha mediante referencia, que acabara siendo una instancia
fecha_fra = date(2000,12,1)
print("la fecha es",fecha_fra)

In [ ]:
#Voy a comprobar si fecha de nacimiento es realmente
fras_enero.info()
#si no tendre que convertir con el metodo to_datetime

In [ ]:
#El accesor dt para usar los metodos de la clase datetime
fras_enero['Mes']= fras_enero['Fecha Nacimiento'].dt.month
fras_enero

Practica 2: Mediante el fichero de excel "Datos telefonia separ...", consolidar los meses de Enero a Junio en un solo
dataframe, crear una columna que me de el "importe factura neto" = Importe factura despues de aplicar un descuento del 8,39% solo
a los clientes que nacieron los dias 1, 5 o 10 del mes correspondiente. Importe factura neto = Importe factura
Queremos tener en una lista las diferentes localidades del campo Localidad
Queremos ver la suma de importe factura neto de los clientes del
servicio ADSL + Movil